# Macro data for commodity ML (FRED)

Run this notebook **from the data_cleaning directory** where `panel_weekly.csv` lives. It creates only new files: `macro/macro_daily_raw.csv`, `macro/macro_weekly.csv`, and `panel_weekly_plus_macro.csv`. It will not modify any existing file.

**Important:** FRED CSV is current vintage, not point-in-time. The notebook shifts weekly macro features by one week as a conservative release buffer, but this does not reconstruct historical revisions. For CPI, unemployment and industrial production, use ALFRED real-time vintages and actual release dates in a subsequent step.


## Step 1 — Configuration


In [ ]:
"""Macro features for commodity return forecasting.

Run from the `data_cleaning` folder of ML_portfolio_and_trading:
    python macro_pipeline.py

Creates NEW files only:
    macro/macro_daily_raw.csv
    macro/macro_weekly.csv
    panel_weekly_plus_macro.csv

Existing panel_weekly.csv, price_features.csv and scores_*.csv are never changed.

Notes:
- FRED Graph CSV gives today's data vintages; it is NOT a point-in-time database.
- Weekly macro features use the PREVIOUS week's data, a conservative availability buffer.
- Before reporting historical claims with revision-prone macro series (CPI/UNRATE/INDPRO),
  use ALFRED real-time vintages and actual release timestamps.
"""

from io import StringIO
from pathlib import Path
import pandas as pd
import requests

# Working directory: data_cleaning, where panel_weekly.csv is saved.
BASE = Path.cwd()
MACRO_DIR = BASE / "macro"
MACRO_DIR.mkdir(exist_ok=True)

SERIES = {
    "DFF": "fed_funds_rate",              # Effective federal funds rate (%)
    "DGS2": "us_2y_yield",                # US 2y Treasury yield (%)
    "DGS10": "us_10y_yield",              # US 10y Treasury yield (%)
    "DTWEXBGS": "usd_trade_weighted",      # Trade-weighted US dollar index
    "VIXCLS": "vix",                      # VIX index
    "T10YIE": "inflation_expectation_10y", # US 10y breakeven inflation rate (%)
}

START = "2009-01-01"




## Step 2 — Download from FRED


In [ ]:
def download_fred_graph_csv(series_id: str, column_name: str) -> pd.DataFrame:
    """Download a publicly accessible time series CSV from a FRED chart.

    The official FRED REST API requires a key, but the FRED chart CSV export
    is public. This is a latest-vintage extract, not historical vintages.
    """
    url = "https://fred.stlouisfed.org/graph/fredgraph.csv"
    response = requests.get(
        url,
        params={"id": series_id, "cosd": START},
        timeout=60,
        headers={"User-Agent": "Mozilla/5.0 (compatible; academic-research/1.0)"},
    )
    response.raise_for_status()
    df = pd.read_csv(StringIO(response.text), na_values=[".", "", "nan"])
    if df.shape[1] != 2:
        raise ValueError(
            f"Unexpected response for {series_id} ({df.columns.tolist()}). "
            "Check the CSV export URL or use the official FRED API with a key."
        )
    df.columns = ["date", column_name]
    df["date"] = pd.to_datetime(df["date"], errors="raise")
    df[column_name] = pd.to_numeric(df[column_name], errors="coerce")
    df = df[df["date"] >= START].drop_duplicates(subset="date")
    if df.empty or df[column_name].notna().sum() == 0:
        raise ValueError(f"No numeric values returned for {series_id}")
    print(f"{series_id:<9} {len(df):>5} rows, {df[column_name].notna().sum():>5} numeric")
    return df.set_index("date")




## Step 3 — Store daily raw data


In [ ]:
def collect_daily_macro() -> pd.DataFrame:
    blocks = [download_fred_graph_csv(sid, name) for sid, name in SERIES.items()]
    daily = pd.concat(blocks, axis=1).sort_index()
    daily.index.name = "date"
    daily.to_csv(MACRO_DIR / "macro_daily_raw.csv")
    print("Saved:", MACRO_DIR / "macro_daily_raw.csv")
    return daily




## Step 4 — Construct Friday-indexed macro features


In [ ]:
def construct_weekly_macro(daily: pd.DataFrame) -> pd.DataFrame:
    # A week is labelled by its Friday, exactly like the user's price/news panel.
    weekly = daily.resample("W-FRI").last().ffill(limit=2)
    features = pd.DataFrame(index=weekly.index)
    features["macro_fed_funds_rate"] = weekly["fed_funds_rate"]
    features["macro_us_10y_yield"] = weekly["us_10y_yield"]
    features["macro_yield_curve_10y_2y"] = (
        weekly["us_10y_yield"] - weekly["us_2y_yield"]
    )
    features["macro_dollar_return_4w"] = weekly["usd_trade_weighted"].pct_change(
        periods=4, fill_method=None
    )
    features["macro_vix"] = weekly["vix"]
    features["macro_inflation_expectation_10y"] = weekly["inflation_expectation_10y"]

    # For the prediction at week t Friday close, use macro measured through
    # Friday t-1, not Friday t: a 1-week buffer for publication delays.
    features = features.shift(1)
    features.index.name = "week"
    features.insert(0, "macro_cutoff_date", features.index - pd.Timedelta(days=7))
    result = features.reset_index()
    result.to_csv(MACRO_DIR / "macro_weekly.csv", index=False)
    print("Saved:", MACRO_DIR / "macro_weekly.csv")
    return result




## Step 5 — Merge into a new file (original files unchanged)


In [ ]:
def merge_into_new_file(macro_weekly: pd.DataFrame) -> pd.DataFrame:
    source = BASE / "panel_weekly.csv"
    if not source.exists():
        raise FileNotFoundError(
            f"Cannot find {source}. Run the script from the data_cleaning folder "
            "containing panel_weekly.csv."
        )
    panel = pd.read_csv(source, parse_dates=["week"])
    assert not panel.duplicated(["week", "commodity"]).any(), (
        "Duplicated panel (week, commodity) keys: inspect before merging"
    )
    assert not macro_weekly["week"].duplicated().any(), (
        "Duplicated macro week keys: inspect before merging"
    )

    merged = panel.merge(macro_weekly, on="week", how="left", validate="many_to_one")
    assert len(merged) == len(panel), "Merge unexpectedly changed number of rows"
    out = BASE / "panel_weekly_plus_macro.csv"
    merged.to_csv(out, index=False)
    new_cols = [c for c in macro_weekly.columns if c.startswith("macro_")]
    print(f"Original panel rows: {len(panel):,}")
    print(f"Enriched panel rows: {len(merged):,}")
    print("Missing macro values by feature:")
    print(merged[new_cols].isna().sum())
    print("Saved NEW file:", out)
    return merged




## Run the pipeline

The `main()` cell runs steps 1–5 and writes *new* outputs. You may inspect `macro_weekly.csv` before running the merge manually if you prefer.


In [ ]:
daily = collect_daily_macro()
weekly = construct_weekly_macro(daily)
display(weekly.head())


In [ ]:
# Run only when you are ready to create the enriched panel:
merged = merge_into_new_file(weekly)
display(merged.head())
